In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType
from delta.tables import DeltaTable

In [0]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
storage_account = "coinwatchsa"

bronze_container = "bronze"
silver_container = "silver"

bronze_root = f"abfss://{bronze_container}@{storage_account}.dfs.core.windows.net/binance/klines/"
silver_path = f"abfss://{silver_container}@{storage_account}.dfs.core.windows.net/candles"

silver_table = "coinwatch.silver.candles"

dbutils.widgets.text("run_date", "2026-06-10")


In [0]:
display(dbutils.fs.ls(f"abfss://bronze@coinwatchsa.dfs.core.windows.net/"))
display(dbutils.fs.ls(f"abfss://silver@coinwatchsa.dfs.core.windows.net/"))
display(dbutils.fs.ls(bronze_root))

In [0]:

run_date = dbutils.widgets.get("run_date").strip()

datetime.strptime(run_date, "%Y-%m-%d")

print("Processing run_datye:", run_date)

In [0]:
kline_schema = StructType([
    StructField("open_time_us", LongType(), False),
    StructField("open", DoubleType(), True),
    StructField("high", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("close_time_us", LongType(), True),
    StructField("quote_volume", DoubleType(), True),
    StructField("trade_count", LongType(), True),
    StructField("taker_buy_base_volume", DoubleType(), True),
    StructField("taker_buy_quote_volume", DoubleType(), True),
    StructField("ignore", StringType(), True),
])

In [ ]:
#create variable bronze path with date widget
bronze_path = f"{bronze_root}symbol=*/date={run_date}"

try:
    from pyspark.errors import AnalysisException
except ImportError:
    from pyspark.sql.utils import AnalysisException

try:
    bronze_df = (
        spark.read
        .option("header", "false")
        .option("basePath", bronze_root)
        .schema(kline_schema)
        .csv(bronze_path)
        .withColumn("source_file", F.col("_metadata.file_path"))
    )
    has_input = bronze_df.limit(1).count() > 0
except AnalysisException:
    # No bronze folder/files landed for this run_date (e.g. Binance had no file).
    has_input = False

In [ ]:
# No input for this run_date is a valid, non-error outcome: exit the job cleanly
# so downstream steps and reruns don't fail on empty days.
if not has_input:
    dbutils.notebook.exit(
        f"SUCCESS: no bronze input for run_date={run_date}; nothing to process."
    )

In [0]:
display(bronze_df.select("symbol", "date", "open_time_us", "open", "high", "low", "close").limit(10))

In [0]:
silver_shaped = (
    bronze_df
    .withColumn("date", F.to_date("date"))
    .withColumn(
        "open_time",
        F.to_timestamp(
            F.from_unixtime((F.col("open_time_us") / F.lit(1000000)).cast("long"))
        )
    )
    .withColumn(
        "close_time",
        F.to_timestamp(
            F.from_unixtime((F.col("close_time_us") / F.lit(1000000)).cast("long"))
        )
    )
    .withColumn("interval", F.lit("1m"))
    .withColumn("ingested_at", F.current_timestamp())
    .drop("ignore")
)

In [0]:
silver_shaped.printSchema()
display(silver_shaped.select("symbol", "date", "open_time", "open", "high", "low", "close", "volume").limit(10))

In [0]:
from pyspark.sql.window import Window
dedup_window = (
    Window
    .partitionBy("symbol", "open_time")
    .orderBy(F.col("source_file").desc())
)
silver_deduped = (
    silver_shaped
    .withColumn("rn",F.row_number().over(dedup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [0]:
duplicate_keys = ( silver_deduped .groupBy("symbol", "open_time") .count() .filter(F.col("count") > 1) )

In [0]:

# A row without symbol or open_time cannot be merged safely.
bad_price_rows = silver_deduped.filter(F.col("high") < F.col("low")) 
bad_price_count = bad_price_rows.count() 
if bad_price_count > 0: raise ValueError(f"{bad_price_count} rows have null symbol/open_time — fix bronze, do not merge")

In [0]:
# Duplicate Keys count of rows, should be 1440
rows_per_day = ( silver_deduped .groupBy("symbol", "date") .count() .orderBy("symbol", "date") )
display(rows_per_day)

In [0]:
# Price Sanity
bad_price_rows = silver_deduped.filter(F.col("high") < F.col("low")) 
bad_price_count = bad_price_rows.count()

if bad_price_count > 0: 
    display(bad_price_rows)
    raise ValueError(f"{bad_price_count} rows have high < low — schema or source is wrong")

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS coinwatch.silver")

# Bootstrap the Delta table once (empty) so the MERGE below has a target.
# Do NOT full-overwrite here: that would wipe every other run_date on each run.
# Rerun idempotency is handled by the MERGE on (symbol, open_time) in the next cell.
if not DeltaTable.isDeltaTable(spark, silver_path):
    (
        silver_deduped.limit(0)
        .write
        .format("delta")
        .partitionBy("date")
        .save(silver_path)
    )

In [0]:
spark.sql(f""" CREATE TABLE IF NOT EXISTS {silver_table} USING DELTA LOCATION '{silver_path}' """)

In [0]:
#MERGE Into Silver - Register the batch as a temp view

silver_deduped.createOrReplaceTempView("silver_batch")

#Run the merge:

spark.sql(f"""
MERGE INTO {silver_table} AS target
USING silver_batch AS source
  ON target.symbol = source.symbol
 AND target.open_time = source.open_time
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
display(spark.sql(f"""
SELECT count(*) AS rows
FROM {silver_table}
WHERE date = DATE('{run_date}')
"""))